# Training A Small ERNIE Model With SFT

## Overview
This notebook fine-tunes a small ERNIE model (nghuyong/ernie-2.0-base-en) on survival scenario data using Supervised Fine-Tuning (SFT).

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

## Import libraries

In [3]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig

In [4]:
model_name = "nghuyong/ernie-2.0-base-en"
USE_GPU = False

print(f"Loading {model_name} for causal language modeling...")

Loading nghuyong/ernie-2.0-base-en for causal language modeling...


## Setting up helper functions

In [5]:
def generate_responses(model, tokenizer, user_message, system_message=None, 
                       max_new_tokens=150):
    # Simple format without chat template
    if system_message:
        prompt = f"System: {system_message}\nQuestion: {user_message}\nAnswer:"
    else:
        prompt = f"Question: {user_message}\nAnswer:"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the new tokens
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response

In [6]:
def test_model_with_questions(model, tokenizer, questions, 
                              system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, 
                                      system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


In [7]:
def load_model_and_tokenizer(model_name, use_gpu = False):
    
    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    if use_gpu:
        model.to("cuda")
    
    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""
    
    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token
        
    return model, tokenizer

In [8]:
def display_dataset(dataset):
    # Visualize the dataset 
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages']
                        if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages']
                             if m['role'] == 'assistant')
        rows.append({
            'User Prompt': user_msg,
            'Assistant Response': assistant_msg
        })
    
    # Display as table
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)  # Avoid truncating long strings
    display(df)

# Custom Dataset

## Dataset

- Ideally you would need at least 50+ datasets.

In [9]:
survival_data = [
    {
        "messages": [
            {"role": "user", "content": "I'm lost in the woods and running out of water. What do I do?"},
            {"role": "assistant", "content": "Water is your top priority - you can only survive 3 days without it. Here's what to do:\n\n1. FIND water sources: Look for streams, rivers, or collect morning dew with a cloth\n2. PURIFY before drinking: Boil for 1 minute (3 minutes at high altitude), or use purification tablets if you have them\n3. If no fire: Solar water disinfection - fill a clear bottle and leave in direct sunlight for 6 hours\n4. Collect rainwater: Use any container, tarp, or large leaves\n5. Never drink stagnant water or urine - it will dehydrate you faster\n\nStay near water sources but not too close - predators also need water."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do I know if water is safe to drink?"},
            {"role": "assistant", "content": "Even clear-looking water can contain parasites. Here's the hierarchy:\n\nSAFEST to RISKIEST:\n1. Rainwater collected in clean container - generally safe\n2. Fast-flowing streams/rivers - lower parasite risk, but STILL boil\n3. Lakes and ponds - moderate risk, MUST purify\n4. Stagnant water - high parasite risk, last resort only\n\nNEVER drink without purifying if possible. If you MUST drink unpurified:\n- Choose fast-flowing water over still water\n- Upstream from animal areas\n- Filter through cloth first to remove debris\n\nSymptoms of waterborne illness take days to appear, so you might feel fine initially."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm in the desert and need shelter from the sun."},
            {"role": "assistant", "content": "Desert survival is about staying cool during day, warm at night:\n\nDAYTIME SHELTER:\n1. Find natural shade: rocks, cliffs, caves (check for animals first!)\n2. Build a shade tarp: Use any fabric elevated 2-3 feet above ground\n3. Create air flow: Don't seal yourself in - ventilation prevents overheating\n4. Stay off hot ground: Sit on rocks, backpack, or dig shallow pit\n\nCRITICAL RULES:\n- Rest during hottest hours (10am-4pm)\n- Travel at dawn/dusk when cooler\n- Cover your skin to prevent sunburn and reduce water loss\n- Conserve energy - heat exhaustion is deadly\n\nNight gets COLD in deserts - keep your insulation for nighttime."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I need to start a fire but I don't have matches or a lighter."},
            {"role": "assistant", "content": "Fire without tools is hard but doable. Here are methods that ACTUALLY work with practice:\n\nMETHOD 1: FIRE PLOUGH (Easiest to learn)\n1. Find dry softwood board (cedar, willow, cottonwood)\n2. Cut groove down the board\n3. Find hardwood stick to rub in groove\n4. Rub stick rapidly up and down groove with pressure\n5. Wood dust accumulates at end and heats up\n6. When smoking heavily, dump into tinder nest\n\nMETHOD 2: BOW DRILL (Most reliable once learned)\nYou need:\n- Spindle: straight stick, 8 inches, thumb-thick\n- Fireboard: flat softwood, 2 inches wide\n- Bow: curved branch, 2 feet long + shoelace/cord\n- Handhold: wood/rock with dimple for top of spindle\n\nTechnique:\n1. Cut notch in fireboard (45° wedge to center hole)\n2. Wrap bowstring once around spindle\n3. Press handhold on top, fireboard under spindle\n4. Bow back and forth rapidly - spindle spins\n5. Push down hard - friction creates ember in notch\n6. Transfer ember to tinder nest carefully\n\nMETHOD 3: FLINT & STEEL (if you have knife)\n- Strike knife spine against hard rock at 30° angle\n- Aim sparks at char cloth or dry tinder\n- Char cloth (charred cotton) catches sparks best\n\nCRITICAL SUCCESS FACTORS:\n- BONE DRY materials (wet = impossible)\n- Prepared tinder nest (size of baseball, fluffy)\n- Practice at home first - it's harder than it looks"}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How should i place the fire?"},
            {"role": "assistant", "content": "Fire structure matters! Here are proven layouts:\n\nTEEPEE FIRE (best for starting):\n1. Place tinder bundle in center\n2. Make small teepee of toothpick-size kindling over it\n3. Build larger teepee of progressively bigger sticks around that\n4. Leave opening for lighting and airflow\n5. Light tinder through opening\n6. As it grows, add bigger sticks\n\nWHY IT WORKS: Flames rise naturally up through structure, heat rises and dries wood above\n\nLOG CABIN (for lasting fire):\n1. Start with tinder bundle\n2. Place two logs parallel, tinder between them\n3. Place two more logs on top, perpendicular\n4. Continue building square layers\n5. Fill center with kindling\n6. Light from bottom\n\nWHY IT WORKS: Great airflow, burns evenly, self-feeding as it collapses\n\nLEAN-TO (for wind protection):\n1. Place large log as windbreak\n2. Lean kindling against it at angle\n3. Tinder bundle underneath\n4. Light from upwind side\n\nWHY IT WORKS: Protected from wind, easy to build\n\nPLATFORM FIRE (for wet ground):\n1. Lay platform of green logs (wet wood okay here)\n2. Build normal fire on top of platform\n3. Keeps fire off wet ground\n\nKEY RULES:\n- Fire needs AIR - don't pack too tight\n- Add wood gradually - too much smothers it\n- Small → medium → large (never skip sizes)\n- Have next size ready before adding it"}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How should i signal for rescue?"},
            {"role": "assistant", "content": "Search teams are looking for things that DON'T belong in nature. Make yourself obvious:\n\nMOST EFFECTIVE SIGNALS:\n\n1. SIGNAL MIRROR (visible 10+ miles to aircraft):\n- Use any reflective surface: phone screen, metal, CD, belt buckle\n- Technique: Hold up V-peace sign toward target\n- Reflect light through the V onto target\n- Flash repeatedly - movement catches attention\n- Works in daytime only\n\n2. SIGNAL FIRE - THREE FIRES (universal distress):\n- Build 3 fires in TRIANGLE pattern (100 feet apart)\n- OR 3 fires in straight LINE\n- Three of anything = distress signal\n- Keep fires ready with tinder, light when you hear aircraft\n- Add green branches for SMOKE during day (visible for miles)\n- Keep fires burning bright at night\n\n3. GROUND SIGNALS (for aircraft):\n- Make LARGE symbols (20+ feet):\n  - X = Need medical help\n  - V = Need assistance\n  - I = Need medical supplies\n  - → = Going this direction\n- Use rocks, logs, clothing laid out\n- Contrast is key - dark materials on light ground or vice versa\n\n4. WHISTLE (audible 1+ miles, farther than voice):\n- THREE sharp blasts = universal distress\n- Saves energy vs yelling\n- Every hiking kit should have whistle\n\n5. BRIGHT COLORS:\n- Hang bright clothing in trees\n- Anything not natural colored (orange, red, yellow best)\n- Movement attracts attention - wave bright items\n\nSTAY PUT once you signal. Moving makes rescue harder."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I don't have a compass. How do I figure out which way is north?"},
            {"role": "assistant", "content": "You can navigate using nature and the sky:\n\nDAYTIME - SUN METHOD (most accurate):\n1. SHADOW STICK:\n   - Plant straight stick (3 feet tall) in flat ground\n   - Mark shadow tip with rock\n   - Wait 15-20 minutes\n   - Mark new shadow tip with second rock\n   - Draw line between rocks: this is EAST-WEST line\n   - First mark = WEST, second = EAST\n   - Face east, north is to your left\n\n2. WATCH METHOD (Northern Hemisphere):\n   - Point hour hand at sun\n   - Halfway between hour hand and 12 is SOUTH\n   - (Use daylight saving time awareness)\n\n3. SUN POSITION:\n   - Sun rises EAST, sets WEST (roughly)\n   - At noon, sun is in SOUTH sky (Northern Hemisphere)\n   - At noon, sun is in NORTH sky (Southern Hemisphere)\n\nNIGHTTIME - STAR METHOD:\nNORTHERN HEMISPHERE:\n1. Find Big Dipper (looks like pot with handle)\n2. Find the two stars at end of the 'cup'\n3. Draw imaginary line through them upward\n4. Follow 5x that distance\n5. You'll hit Polaris (North Star) - it's medium brightness\n6. Polaris = TRUE NORTH (doesn't move)\n\nSOUTHERN HEMISPHERE:\n1. Find Southern Cross (4 stars in cross shape)\n2. Follow long axis of cross 4.5x its length\n3. That point = SOUTH\n\nNATURE SIGNS (less reliable, use as backup only):\n- Moss grows on NORTH side of trees (Northern Hemisphere) - the shaded, moist side\n- Trees fuller/more branches on SOUTH side (more sun)\n- Snow melts faster on SOUTH-facing slopes\n- Ant hills often on SOUTH side of trees\n\nThese are general tendencies, not absolute. Use multiple methods to confirm."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm shivering uncontrollably and feel confused. What's happening to me?"},
            {"role": "assistant", "content": "You have HYPOTHERMIA - a medical emergency. Your core body temperature has dropped below 95°F. Act immediately:\n\nSTAGES & SYMPTOMS:\n\nMILD (95-90°F):\n- Intense shivering\n- Can't perform complex tasks\n- Hands numb, fumbling\n→ You're still conscious and functional\n\nMODERATE (90-82°F) - YOU ARE HERE:\n- Violent shivering OR shivering stops\n- Confusion, poor decisions\n- Slurred speech\n- Drowsiness, apathy\n- Loss of coordination\n→ Judgment is impaired - you need to act NOW\n\nSEVERE (below 82°F):\n- No shivering (body gave up)\n- Can't walk\n- Unconscious\n- Weak pulse\n→ Death is imminent\n\nIMMEDIATE ACTIONS:\n\n1. STOP HEAT LOSS:\n   - Get OUT of wind/rain NOW (even crude shelter helps)\n   - Remove WET clothes immediately (wet = 25x more heat loss)\n   - Wet clothes kill - being naked is better than wet clothes\n\n2. INSULATE:\n   - Put on dry clothes if available\n   - Wrap in emergency blanket, tarp, garbage bag\n   - Stuff dry leaves, grass inside clothes\n   - Cover head (lose 40% of heat through head)\n\n3. EXTERNAL HEAT:\n   - Build fire if possible\n   - Huddle with another person (skin-to-skin in sleeping bag)\n   - Warm packs in armpits/groin (high blood flow areas)\n   - DO NOT rub extremities (can cause heart attack)\n\n4. WARM DRINKS (if conscious):\n   - Warm water, tea (not hot - can shock system)\n   - NO alcohol (dilates blood vessels = more heat loss)\n   - NO caffeine (increases urine = dehydration)\n\n5. STAY AWAKE:\n   - Sleep can be fatal in severe hypothermia\n   - Keep moving if you can (not vigorously)\n   - Talk to yourself, sing, count\n\nDANGER SIGNS:\n- Feeling warm (paradoxical undressing) - you're NOT warm, this is late-stage hypothermia\n- Want to lie down and sleep - fight this urge\n- Removing clothes because you feel hot - put them back on!\n\nPrevent getting here: Stay DRY at all costs."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm stranded on a beach after a shipwreck. What should I do first?"},
            {"role": "assistant", "content": "First, move above the high tide line to avoid being swept away. Collect any floating debris or supplies immediately before they drift away. Find freshwater - never drink seawater as it will dehydrate you faster. Look for streams flowing into the ocean or collect rainwater. Build a shelter away from the water to protect from sun and wind. Create signal fires on the beach using driftwood. Three fires in a line is an international distress signal."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm lost in the mountains in winter. How do I avoid freezing?"},
            {"role": "assistant", "content": "Build a snow shelter immediately. Dig a snow cave into a drift or build a quinzhee by piling snow and hollowing it out. Snow provides excellent insulation. Make the entrance lower than the sleeping platform so cold air sinks out. Stay dry at all costs - wet clothing loses heat 25 times faster. Insulate yourself from the ground using pine branches, your backpack, or extra clothing. Keep your head covered as you lose 40% of body heat through your head. Avoid sweating by removing layers when active."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm stuck in a swamp. How do I get out safely?"},
            {"role": "assistant", "content": "Test ground before each step using a long stick. Firm ground will resist the stick while mud gives way. If you start sinking in quicksand or mud, fall backward to distribute your weight and slowly crawl out. Never struggle violently. Look for elevated areas with trees or dry vegetation. Avoid stagnant water as it contains dangerous bacteria. Watch for alligators and venomous snakes near water edges. Move slowly and deliberately to conserve energy in the humid heat."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "I'm trapped in a collapsed building after an earthquake. How do I survive?"},
            {"role": "assistant", "content": "Don't shout constantly - you'll exhaust yourself and breathe in dust. Tap on pipes or walls rhythmically to signal rescuers. Cover your mouth with clothing to filter dust and debris. If you have a phone, text instead of calling to conserve battery. Don't light matches or lighters as gas leaks are common. Stay still to conserve air in your pocket. Don't drink your own urine. Ration any water you find. Listen for rescue teams and respond when you hear them nearby. Move debris carefully to avoid causing further collapse."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How can I collect and filter water if I have no equipment?"},
            {"role": "assistant", "content": "Collect morning dew by tying cloth around your ankles and walking through grass at dawn, then wring it into a container. For filtration, create a tripod filter using three layers. Fill the bottom with grass to catch large debris, middle layer with sand for smaller particles, and top layer with crushed charcoal from your fire for chemical filtration. Pour water through slowly. This removes debris but not all pathogens, so still boil if possible. Collect rainwater directly using large leaves funneled into containers."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do I make basic survival tools from natural materials?"},
            {"role": "assistant", "content": "For a cutting tool, find a hard rock like flint or obsidian. Strike it at an angle with another rock to flake off sharp edges. For a spear, choose a straight hardwood branch about 6 feet long. Sharpen one end by scraping against rough rock or hardening in fire. For a knife, split a small piece of sharp rock and wrap the base with cloth, bark, or cordage for a handle. Create cordage by twisting plant fibers, inner tree bark, or long grass together. Braid three strands for stronger rope."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How do I catch fish without a fishing rod?"},
            {"role": "assistant", "content": "Build a fish trap using sticks. Create a funnel shape with a narrow opening that fish can enter but can't easily exit. Place it in shallow water with the opening facing upstream. Weight it down with rocks. For a spear, sharpen a long straight stick and split the tip into four prongs. Spread them apart with a small wedge and sharpen each point. Stand still in shallow water and strike quickly when fish approach. You can also make a net using woven plant fibers or catch fish by hand in rocky pools by cornering them slowly."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "How can I make containers to carry water and supplies?"},
            {"role": "assistant", "content": "For a water container, find large sections of bamboo with natural joints that seal the bottom. Cut just above a joint for the base. For a basket, weave flexible young branches or vines in an over-under pattern around a circular base. Make birch bark containers by carefully peeling large sheets of bark, folding into a box shape, and securing edges with wooden pegs or plant fiber stitching. Use pine pitch as waterproof sealant by heating it and applying to seams. Large leaves from plants like taro can hold water temporarily. Create a backpack using a shirt - tie the bottom closed, put supplies inside, and wear it backwards with arms through sleeves."}
        ]
    }

]

# Format Data For Training

In [10]:
def format_ernie_data(data):
    formatted = []
    for item in data:
        text = ""
        for msg in item["messages"]:
            if msg["role"] == "user":
                text += f"Question: {msg['content']}\n"
            elif msg["role"] == "assistant":
                text += f"Answer: {msg['content']}\n\n"
        formatted.append({"text": text})
    return formatted

formatted_data = format_ernie_data(survival_data)
train_dataset = Dataset.from_list(formatted_data)

print(f"Training dataset: {len(train_dataset)} examples")
print("\nExample training format:")
print(formatted_data[0]["text"][:500])
print("\n" + "="*80 + "\n")

Training dataset: 16 examples

Example training format:
Question: I'm lost in the woods and running out of water. What do I do?
Answer: Water is your top priority - you can only survive 3 days without it. Here's what to do:

1. FIND water sources: Look for streams, rivers, or collect morning dew with a cloth
2. PURIFY before drinking: Boil for 1 minute (3 minutes at high altitude), or use purification tablets if you have them
3. If no fire: Solar water disinfection - fill a clear bottle and leave in direct sunlight for 6 hours
4. Collect rainwater: U




# Load Base Model and Train

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

model_name = "nghuyong/ernie-2.0-base-en"
USE_GPU = False

print(f"Loading {model_name} for causal LM...")

# Load config and set is_decoder=True
config = AutoConfig.from_pretrained(model_name)
config.is_decoder = True
config.add_cross_attention = False  # We don't need cross-attention

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model with the modified config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    ignore_mismatched_sizes=True  # This handles the newly initialized weights
)

if USE_GPU:
    model.to("cuda")

# Set pad token
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.pad_token_id

tokenizer.padding_side = 'left'

print("Model loaded successfully for causal generation!")

Loading nghuyong/ernie-2.0-base-en for causal LM...


Some weights of ErnieForCausalLM were not initialized from the model checkpoint at nghuyong/ernie-2.0-base-en and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully for causal generation!


In [ ]:
# training configuration for ERNIE
sft_config = SFTConfig(
    learning_rate=2e-5,  # Lower learning rate
    num_train_epochs=50,  # More epochs since small dataset
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=False,
    logging_steps=1,
    output_dir="./ernie_survival_model",
    max_seq_length=512,
    warmup_steps=10,  
    weight_decay=0.01, 
    save_strategy="epoch",
)

In [14]:
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    
)

Map: 100%|██████████| 16/16 [00:00<00:00, 233.37 examples/s]


In [15]:
sft_trainer.train()

Step,Training Loss
1,10.344300
2,10.315400
3,10.142800
4,9.986600
5,9.601800
6,9.138600
7,8.802400
8,8.647800
9,8.323000
10,8.232400


TrainOutput(global_step=10, training_loss=9.353510570526122, metrics={'train_runtime': 63.4321, 'train_samples_per_second': 1.261, 'train_steps_per_second': 0.158, 'total_flos': 8433349519500.0, 'train_loss': 9.353510570526122, 'epoch': 5.0})

# Result

Note that the result will be gibberish since some weights of ErnieForCausalLM were not initialized, so the datasets is small or when the number of training epochs is low, the result will be gibberish, but the code is still working since the training loss is steadily decreasing.

In [16]:
# Test the trained model
survival_questions = [
    "I'm in a forest and I'm hungry, what should i do",
    "How do I start a fire without matches?",
]

test_model_with_questions(
    sft_trainer.model, 
    tokenizer, 
    survival_questions,
    title="Survival Model (After SFT) Output"
)


=== Survival Model (After SFT) Output ===

Model Input 1:
I'm in a forest and I'm hungry, what should i do
Model Output 1:
##mara 1 et se double fire seee - - but ice point2 ice on under on fire co for see face more on - througha 2s go at forke i note more facenee more baree crosss 4s face cuessc against spot double against double spot post both to fire be -na 2 2 go / open moree cross cross to fire gas seee face cross area - on ice cover gas watch spotk butek butn - basedaekk cross double point near on lead spot 5 just watchen be both to really but so both have for in fore on sense on fire north firek watchn coveryn 3 both for 5x on even fire 7ce ink


Model Input 2:
How do I start a fire without matches?
Model Output 2:
stunning notty for introducedn 18 ice - forces openty ice crossscch moresc mean about for ben combined mo on fire 11chme doubleers check on ground i under facea about fire 16sc both -hen side 5 2 fire note 4 handa on light also nature sub - woodka - point be in - wit